Notebook to calculate the bedmodes, saturated sand flux parameters (Courrech du Pont et al. 2024) and find the optimal threshold speed for fixed dune slopes

### Imports & dask client

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import pandas as pd
import geopandas as gpd
from shapely import Point
import time

from dask.distributed import Client
client = Client(processes=False,threads_per_worker=4) # process=False helped running code below on bigboy
client

### Functions and definitions

In [ ]:
def _elongationmode(x,y):
    try:
        temp0 = CubicSpline(x,y,bc_type='periodic')
    except: # https://github.com/scipy/scipy/blob/v1.16.2/scipy/interpolate/_cubic.py#L919
        temp0 = CubicSpline(x,np.hstack((y[-1],y[1:])),bc_type='periodic')
    temp1 = temp0.roots(extrapolate=False)
    return temp1[np.argmin(temp0.derivative()(temp1))]

def elongationmode(ds,dim):
    return xr.apply_ufunc(
        _elongationmode,
        ds[dim],ds,
        input_core_dims=[[dim],[dim]],output_core_dims=[[]],
        vectorize=True,dask='parallelized',output_dtypes=[float])
    
def _bedmode(x,y):
    try:
        temp0 = CubicSpline(x,y,bc_type='periodic')
    except: # https://github.com/scipy/scipy/blob/v1.16.2/scipy/interpolate/_cubic.py#L919
        temp0 = CubicSpline(x,np.hstack((y[-1],y[1:])),bc_type='periodic')
    temp1 = np.hstack((temp0.derivative().roots(extrapolate=False),x[0],x[-1]))
    return temp1[np.argmax(temp0(temp1))]

def bedmode(ds,dim):
    return xr.apply_ufunc(
        _bedmode,
        ds[dim],ds,
        input_core_dims=[[dim],[dim]],output_core_dims=[[]],
        vectorize=True,dask='parallelized',output_dtypes=[float])

def attributes_bedmodes(ds):
    ds['longitude'].attrs['long_name'] = 'Longitude $\\lambda$'
    ds['longitude'].attrs['units'] = '$\\degree$'
    ds['longitude'].attrs['description'] = 'Longitude $\\lambda$ in degrees.'
    ds['latitude'].attrs['long_name'] = 'Latitude $\\phi$'
    ds['latitude'].attrs['units'] = '$\\degree$'
    ds['latitude'].attrs['description'] = 'Latitude $\\phi$ in degrees.'
    ds['ut'].attrs['long_name'] = 'Threshold friction velocity $u_{\\ast,th}$'
    ds['ut'].attrs['units'] = 'm/s'
    ds['ut'].attrs['description'] = 'Threshold friction velocity $u_{\\ast,th}$ in m/s.'
    ds['da'].attrs['long_name'] = 'Signed ERA5-Land cell average dune crest orientation $\\alpha_{obs}$'
    ds['da'].attrs['units'] = 'rad'
    ds['da'].attrs['description'] = 'ERA5-Land cell average dune crest orientation $\\alpha_{obs}$ in rad. Unsigned averaged orientation was extracted from 100 m segments of crest lines in cell and signed by the average orientation of y-junction bisectors (at 100 m radius) in cell.'
    ds['dm'].attrs['long_name'] = 'ERA5-Land cell fixed dune aspect ratio $H/L$.'
    ds['dm'].attrs['units'] = 'unitless'
    ds['dm'].attrs['description'] = 'ERA5-Land cell average dune aspect (fixed) ratio $H/L$. Ratio was defined at half dune height from shallower trough to crest and dune width at half height. Transects with a 100 m spacing perpendiculat to crest lines where used to obtain profiles.'
    ds['ab'].attrs['long_name'] = 'Migration mode unsigned direction $\\alpha_H$'
    ds['ab'].attrs['units'] = 'rad'
    ds['ab'].attrs['description'] = 'Migration mode unsigned direction $\\alpha_H$ in radians.'
    ds['ae'].attrs['long_name'] = 'Elongation mode signed direction $\\alpha_E$'
    ds['ae'].attrs['units'] = 'rad'
    ds['ae'].attrs['description'] = 'Elongation mode signed direction $\\alpha_E$ in radians. The direction longitudinal crests elongate.'
    ds['dabdut'].attrs['long_name'] = 'Derivative of the migration mode unsigned direction $\\partial\\alpha_H$/$\\partial u_{\\ast,th}$'
    ds['dabdut'].attrs['units'] = '$\\degree$/ms$^{-1}$'
    ds['dabdut'].attrs['description'] = 'Derivative of the migration mode unsigned direction $\\alpha_H$ with respect to $u_{\\ast,th}$.'
    ds['daedut'].attrs['long_name'] = 'Derivative of the elongation mode unsigned direction $\\partial\\alpha_E$/$\\partial u_{\\ast,th}$'
    ds['daedut'].attrs['units'] = '$\\degree$/ms$^{-1}$'
    ds['daedut'].attrs['description'] = 'Derivative of the elongation mode unsigned direction $\\alpha_E$ with respect to $u_{\\ast,th}$.'

    ds.attrs['Wind data'] = 'ERA5 $u_{10}$ and $v_{10}$ interpolated to ERA5-Land grid, 1950-2025 inclusive.'
    ds.attrs['Constants'] = f'Measured wind height $z$ = {z} m. Roughness length $z_0$ = {z0} m. Von Karman constant $\\kappa$ = {kappa}. Speed-up parameter $\\beta$ = {int(beta)}. Gravity $g$ = {g} m/s$^2$. Sediment density $\\rho_s$ = {rho_s} kg/m$^3$. Fluid density $\\rho_f$ = {rho_f} kg/m$^3$. Flux constant $a_q$ = {fluxconstant}.'
    ds.attrs['Formulas'] = 'See Courrech du Pont et al. Earth-Science Reviews (2024), https://doi.org/10.1016/j.earscirev.2024.104772.'
    ds.attrs['Notes'] = 'Bedmodes were calculated for fixed dune aspect ratios ($dm$). $\\beta$ was defined in such a way that it compensates for our definitions of $H/L$ and aligns with Courrech du Pont et al. Earth-Science Reviews (2024), https://doi.org/10.1016/j.earscirev.2024.104772. All fluxes are saturated and adjusted for porosity.'
    ds.attrs['Author'] = 'Dominik Nommensen$^1$, Andrew Gunn$^1$. $^1$Monash University.'
    ds.attrs['date_modified'] = time.strftime("%Y/%m/%d")

    return ds

def attributes_qsat(ds):
    ds['longitude'].attrs['long_name'] = 'Longitude $\\lambda$'
    ds['longitude'].attrs['units'] = '$\\degree$'
    ds['longitude'].attrs['description'] = 'Longitude $\\lambda$ in degrees.'
    ds['latitude'].attrs['long_name'] = 'Latitude $\\phi$'
    ds['latitude'].attrs['units'] = '$\\degree$'
    ds['latitude'].attrs['description'] = 'Latitude $\\phi$ in degrees.'
    ds['ut'].attrs['long_name'] = 'Threshold friction velocity $u_{\\ast,th}$'
    ds['ut'].attrs['units'] = 'm/s'
    ds['ut'].attrs['description'] = 'Threshold friction velocity $u_{\\ast,th}$ in m/s.'
    ds['da'].attrs['long_name'] = 'Signed ERA5-Land cell average dune crest orientation $\\alpha_{obs}$'
    ds['da'].attrs['units'] = 'rad'
    ds['da'].attrs['description'] = 'ERA5-Land cell average dune crest orientation $\\alpha_{obs}$ in rad. Unsigned averaged orientation was extracted from 100 m segments of crest lines in cell and signed by the average orientation of y-junction bisectors (at 100 m radius) in cell.'
    ds['dm'].attrs['long_name'] = 'ERA5-Land cell average dune aspect (fixed) ratio $H/L$.'
    ds['dm'].attrs['units'] = 'unitless'
    ds['dm'].attrs['description'] = 'ERA5-Land cell average dune aspect (fixed) ratio $H/L$. Ratio was defined at half dune height from shallower trough to crest and dune width at half height. Transects with a 100 m spacing perpendiculat to crest lines where used to obtain profiles.'
    ds['ab'].attrs['long_name'] = 'Migration mode unsigned direction $\\alpha_H$'
    ds['ab'].attrs['units'] = 'rad'
    ds['ab'].attrs['description'] = 'Migration mode unsigned direction $\\alpha_H$ in radians.'
    ds['ae'].attrs['long_name'] = 'Elongation mode signed direction $\\alpha_E$'
    ds['ae'].attrs['units'] = 'rad'
    ds['ae'].attrs['description'] = 'Elongation mode signed direction $\\alpha_E$ in radians. The direction longitudinal crests elongate.'
    ds['dabdut'].attrs['long_name'] = 'Derivative of the migration mode unsigned direction $\\partial\\alpha_H$/$\\partial u_{\\ast,th}$'
    ds['dabdut'].attrs['units'] = '$\\degree$/ms$^{-1}$'
    ds['dabdut'].attrs['description'] = 'Derivative of the migration mode unsigned direction $\\alpha_H$ with respect to $u_{\\ast,th}$.'
    ds['daedut'].attrs['long_name'] = 'Derivative of the elongation mode unsigned direction $\\partial\\alpha_E$/$\\partial u_{\\ast,th}$'
    ds['daedut'].attrs['units'] = '$\\degree$/ms$^{-1}$'
    ds['daedut'].attrs['description'] = 'Derivative of the elongation mode unsigned direction $\\alpha_E$ with respect to $u_{\\ast,th}$.'

    ds['at'].attrs['long_name'] = 'Migration signed direction $\\alpha_T$'
    ds['at'].attrs['units'] = 'rad'
    ds['at'].attrs['description'] = 'Migration signed direction $\\alpha_T$ in radians. The direction transverse crests migrate.'
    ds['ao'].attrs['long_name'] = 'Oblique signed direction $\\alpha_O$'
    ds['ao'].attrs['units'] = 'rad'
    ds['ao'].attrs['description'] = 'Oblique signed signed direction $\\alpha_O$ in radians. The direction oblique dune crests elongate.'
    ds['rdpa'].attrs['long_name'] = 'Resultant drift direction $RDD$'
    ds['rdpa'].attrs['units'] = 'rad'
    ds['rdpa'].attrs['description'] = 'Signed resultant drift direction $RDD$ in radians. The direction sand would move on a flat bed.'
    
    ds['dpm'].attrs['long_name'] = 'Drift potential $DP$'
    ds['dpm'].attrs['units'] = 'm$^2$/s'
    ds['dpm'].attrs['description'] = 'Drift potential $DP$ in m$^2$/s. The total flux on a flat bed.'
    ds['rdpm'].attrs['long_name'] = 'Resultant drift potential $RDP$'
    ds['rdpm'].attrs['units'] = 'm$^2$/s'
    ds['rdpm'].attrs['description'] = 'Resultant drift potential $RDP$ in m$^2$/s. The net flux on a flat bed.'

    ds['qnlb'].attrs['long_name'] = 'Net along-crest migration-mode flux $\\langle Q_{\\parallel}(\\alpha_H)\\rangle$'
    ds['qnlb'].attrs['units'] = 'm$^2$/s'
    ds['qnlb'].attrs['description'] = 'Net along-crest migration-mode flux $\\langle Q_{\\parallel}(\\alpha_H)\\rangle$ in m$^2$/s.'
    ds['qncb'].attrs['long_name'] = 'Net across-crest migration-mode flux $\\langle Q_{\\bot}(\\alpha_H)\\rangle$'
    ds['qncb'].attrs['units'] = 'm$^2$/s'
    ds['qncb'].attrs['description'] = 'Net across-crest migration-mode flux $\\langle Q_{\\bot}(\\alpha_H)\\rangle$ in m$^2$/s.'
    ds['qalb'].attrs['long_name'] = 'Absolute along-crest migration-mode flux $\\langle |Q_{\\parallel}(\\alpha_H)|\\rangle$'
    ds['qalb'].attrs['units'] = 'm$^2$/s'
    ds['qalb'].attrs['description'] = 'Absolute along-crest migration-mode flux $\\langle |Q_{\\parallel}(\\alpha_H)|\\rangle$ in m$^2$/s.'
    ds['qacb'].attrs['long_name'] = 'Absolute across-crest migration-mode flux $\\langle |Q_{\\bot}(\\alpha_H)|\\rangle$'
    ds['qacb'].attrs['units'] = 'm$^2$/s'
    ds['qacb'].attrs['description'] = 'Absolute across-crest migration-mode flux $\\langle |Q_{\\bot}(\\alpha_H)|\\rangle$ in m$^2$/s.'
    
    ds['qnle'].attrs['long_name'] = 'Net along-crest elongation-mode flux $\\langle Q_{\\parallel}(\\alpha_E)\\rangle$'
    ds['qnle'].attrs['units'] = 'm$^2$/s'
    ds['qnle'].attrs['description'] = 'Net along-crest elongation-mode flux $\\langle Q_{\\parallel}(\\alpha_E)\\rangle$ in m$^2$/s.'
    ds['qale'].attrs['long_name'] = 'Absolute along-crest elongation-mode flux $\\langle |Q_{\\parallel}(\\alpha_E)|\\rangle$'
    ds['qale'].attrs['units'] = 'm$^2$/s'
    ds['qale'].attrs['description'] = 'Absolute along-crest elongation-mode flux $\\langle |Q_{\\parallel}(\\alpha_E)|\\rangle$ in m$^2$/s.'
    ds['qace'].attrs['long_name'] = 'Absolute across-crest elongation-mode flux $\\langle |Q_{\\bot}(\\alpha_E)|\\rangle$'
    ds['qace'].attrs['units'] = 'm$^2$/s'
    ds['qace'].attrs['description'] = 'Absolute across-crest elongation-mode flux $\\langle |Q_{\\bot}(\\alpha_E)|\\rangle$ in m$^2$/s.'
    
    ds['qha_rdp_dp'].attrs['long_name'] = '$RDP/DP$'
    ds['qha_rdp_dp'].attrs['units'] = '-'
    ds['qha_rdp_dp'].attrs['description'] = 'Resultant drift potential over drift potential $RDP/DP$. Flat bed flux directionality.'
    ds['sige_sigh'].attrs['long_name'] = '$\\sigma_E/\\sigma_H$'
    ds['sige_sigh'].attrs['units'] = '-'
    ds['sige_sigh'].attrs['description'] = 'Stability of elongating dunes on starved beds $\\langle|Q_{\\bot}(\\alpha_E)|\\rangle/\\langle|Q_{\\bot}(\\alpha_H)|\\rangle$.'
    
    ds['qhab_nl_nc'].attrs['long_name'] = '$\\hat{Q}_{\\parallel/\\bot,H}$'
    ds['qhab_nl_nc'].attrs['units'] = '-'
    ds['qhab_nl_nc'].attrs['description'] = 'Straightness vs. sinuosity of periodic long-crested dunes on covered bed (bed instability mode, growth in height dominates), asymmetry of migrating dunes $|\\langle Q_{\\parallel}(\\alpha_H)\\rangle/\\langle Q_{\\bot}(\\alpha_H)\\rangle|$.'
    ds['qhab_ac_nc'].attrs['long_name'] = '$\\hat{Q}_{|\\bot|/\\bot,H}$'
    ds['qhab_ac_nc'].attrs['units'] = '-'
    ds['qhab_ac_nc'].attrs['description'] = 'Relative balance between growth in height and migration dynamics $|\\langle |Q_{\\bot}(\\alpha_H)|\\rangle/\\langle Q_{\\bot}(\\alpha_H)\\rangle|$.'
    
    ds['qhne_ac_nl'].attrs['long_name'] = '$\\hat{Q}_{|\\bot|/\\parallel,E}$'
    ds['qhne_ac_nl'].attrs['units'] = '-'
    ds['qhne_ac_nl'].attrs['description'] = 'Morphology of elongating dunes $\\langle |Q_{\\bot}(\\alpha_E)|\\rangle/\\langle Q_{\\parallel}(\\alpha_E)\\rangle$.'

    ds['ufm'].attrs['long_name'] = '$\\langle u_{\\ast,f}|u_{\\ast,f}>u_{\\ast,th}\\rangle/u_{\\ast,th}$'
    ds['ufm'].attrs['units'] = '-'
    ds['ufm'].attrs['description'] = 'Average above-threshold flat-bed friction velocity, normalised by threshold $\\langle u_{\\ast,f}|u_{\\ast,f}>u_{\\ast,th}\\rangle/u_{\\ast,th}$.'
    ds['uft'].attrs['long_name'] = '$T|u_{\\ast,f}>u_{\\ast,th}/T$'
    ds['uft'].attrs['units'] = '-'
    ds['uft'].attrs['description'] = 'Fraction of time flat-bed friction velocity is above threshold $T|u_{\\ast,f}>u_{\\ast,th}/T$.'
    
    ds['qhbf'].attrs['long_name'] = '$\\langle|Q(\\alpha_H)|\\rangle|u_{\\ast,f}>u_{\\ast,th}/\\langle|Q(\\alpha_H)|\\rangle$'
    ds['qhbf'].attrs['units'] = '-'
    ds['qhbf'].attrs['description'] = 'Fraction of bed-mode flux when flat-bed friction velocity is above threshold $\\langle|Q(\\alpha_H)|\\rangle|u_{\\ast,f}>u_{\\ast,th}/\\langle|Q(\\alpha_H)|\\rangle$.'
    ds['qhef'].attrs['long_name'] = '$\\langle|Q(\\alpha_E)|\\rangle|u_{\\ast,f}>u_{\\ast,th}/\\langle|Q(\\alpha_E)|\\rangle$'
    ds['qhef'].attrs['units'] = '-'
    ds['qhef'].attrs['description'] = 'Fraction of elongation-mode flux when flat-bed friction velocity is above threshold $\\langle|Q(\\alpha_E)|\\rangle|u_{\\ast,f}>u_{\\ast,th}/\\langle|Q(\\alpha_E)|\\rangle$.'
    
    ds.attrs['Wind data'] = 'ERA5 $u_{10}$ and $v_{10}$ interpolated to ERA5-Land grid, 1950-2025 inclusive.'
    ds.attrs['Constants'] = f'Measured wind height $z$ = {z} m. Roughness length $z_0$ = {z0} m. Von Karman constant $\\kappa$ = {kappa}. Speed-up parameter $\\beta$ = {int(beta)}. Gravity $g$ = {g} m/s$^2$. Sediment density $\\rho_s$ = {rho_s} kg/m$^3$. Fluid density $\\rho_f$ = {rho_f} kg/m$^3$. Flux constant $a_q$ = {fluxconstant}.'
    ds.attrs['Formulas'] = 'See Courrech du Pont et al. Earth-Science Reviews (2024), https://doi.org/10.1016/j.earscirev.2024.104772.'
    ds.attrs['Notes'] = 'Bedmodes were calculated for fixed dune aspect ratios ($dm$). $\\beta$ was defined in such a way that it compensates for our definitions of $H/L$ and aligns with Courrech du Pont et al. Earth-Science Reviews (2024), https://doi.org/10.1016/j.earscirev.2024.104772. All fluxes are saturated and adjusted for porosity.'
    ds.attrs['Author'] = 'Dominik Nommensen$^1$, Andrew Gunn$^1$. $^1$Monash University.'
    ds.attrs['date_modified'] = time.strftime("%Y/%m/%d")
    
    return ds

### Load wind and dune topo data (This is now for a fixed aspect ratio)

In [ ]:
# definitions
z            = 10
z0           = 1e-3
kappa        = 0.41
beta         = 6/2 # same as used for the dynamic slope
g            = 9.81
rho_s        = 2650
rho_f        = 1.2
fluxconstant = 5/0.6 # for fraction factor
fluxk        = fluxconstant/g*rho_f/rho_s

# load the era5-land wind data and the dune aspect and orientation for bedmode calculations
chunksize = 64
lonchunks = [45,51]
latchunks = [15,20]

# wind data
dh = xr.open_mfdataset("era5-land_1950_2025_uam10_hist.zarr",engine="zarr",consolidated=False).isel(latitude=slice(latchunks[0]*chunksize,latchunks[1]*chunksize),
                                                                                                    longitude=slice(lonchunks[0]*chunksize,lonchunks[1]*chunksize))
# dune topography
ds = xr.open_dataset("../Data/Netcdfs/aus_dune_topo.nc")
ds = ds.rename_vars({"lon":"longitude","lat":"latitude"})
ds = ds.set_index(index_poly=("latitude", "longitude"))
ds = ds.unstack("index_poly")
ds = ds.reindex(latitude=dh.latitude.values,longitude=dh.longitude.values,fill_value=np.nan,method="nearest",tolerance=0.05)
daspect = float(np.nanmedian(ds.daspect)) # fixed median across continent
dh["daspect"] = (("latitude","longitude"),np.ones_like(ds.avg100sig)*daspect)
dh["daspect"] = dh.daspect.chunk({"latitude": chunksize, "longitude": chunksize}) # dune aspect ratio
dh = dh.assign(da=ds.avg100sig) # signed dune orientation
dh = dh.assign_coords(ut=np.linspace(0.01,0.64,2**6)).assign_coords(do=np.linspace(-np.pi,np.pi,2**7+1)) # set of threshold velocities and dune orientations

# mask of chunks containing dune data (as a function of small_chunksize). Note: Bit redundant for chunksize 1 but versatile if one wants to increase.
small_chunksize=1
llon=len(dh.longitude)//small_chunksize
llat=len(dh.latitude)//small_chunksize
chunkmask=np.zeros((llat,llon))
for j in range(chunkmask.shape[0]):
   for i in range(chunkmask.shape[1]):
       if not np.isnan(dh.da.isel(latitude=slice(j*small_chunksize,(j+1)*small_chunksize),longitude=slice(i*small_chunksize,(i+1)*small_chunksize))).values.all():
           chunkmask[j,i]=1
chunkmask = chunkmask.astype(int)

### Calculate bedmodes

In [ ]:
# calculate the bedmodes for above parameters and extracted dune topology for only dune containing lat/lon
temp = []
counter = 0
start = time.time()
for j in range(chunkmask.shape[0]):
    for i in range(chunkmask.shape[1]):
        if not chunkmask[j,i]:
            continue
        dhtemp = dh.isel(latitude=slice(j*small_chunksize,(j+1)*small_chunksize),longitude=slice(i*small_chunksize,(i+1)*small_chunksize))
        # calculate the elongation and bed mode for chunks with dunes
        last_start = time.time()
        dhtemp = dhtemp.assign(uc=(dhtemp.um_bin*kappa/np.log(z/z0))*(1+beta*dhtemp.daspect*np.abs(np.sin(dhtemp.ua_bin-dhtemp.do))))
        dhtemp = dhtemp.assign(qa=xr.where(dhtemp.uc>dhtemp.ut,(dhtemp.uc**2-dhtemp.ut**2)*dhtemp.ut,np.nan))
        dhtemp = dhtemp.assign(qc=dhtemp.qa*np.sin(dhtemp.ua_bin-dhtemp.do))
        dhtemp = dhtemp.assign(qac=np.abs(dhtemp.qc).weighted(dhtemp.histogram_um_ua).sum(('um_bin','ua_bin')))
        dhtemp = dhtemp.assign(qnc=dhtemp.qc.weighted(dhtemp.histogram_um_ua).sum(('um_bin','ua_bin')))
        dhtemp['elongation'] = elongationmode(dhtemp.qnc,'do')
        dhtemp['bed'] = bedmode(dhtemp.qac.sel(do=slice(0,None)),'do')
        dhtemp_sub = dhtemp[['daspect','da','elongation','bed']] # keeping dune orientatin and aspect for future calculations
        dmodes = xr.merge([dhtemp_sub,dhtemp_sub.differentiate('ut').rename(dict(elongation='delongationdut',bed='dbeddut'))])
        temp.append(dmodes.compute())
        counter+=1
        print('\rdune cells: %4d/%4d, time last: %.2f, time total: %6d'%(counter,chunkmask.sum(),time.time()-last_start,time.time()-start),end='')

# create dataset, relabel, add attributes and save bedmodes
dsout = xr.combine_by_coords(temp)
dsout = dsout.rename_vars({"daspect":"dm","elongation":"ae","bed":"ab","delongationdut":"daedut","dbeddut":"dabdut"})
dsout = attributes_bedmodes(dsout) # update attributes
dsout.to_netcdf("era5-land_1950_2025_bedmodes_fixed_dm.nc")

### Calculate qsat params for dynamic dm

In [ ]:
# definitions
z            = 10
z0           = 1e-3
kappa        = 0.41
beta         = 6/2 #same as used for the dynamic slope
g            = 9.81
rho_s        = 2650
rho_f        = 1.2
fluxconstant = 5/0.6 # for fraction factor
fluxk        = fluxconstant/g*rho_f/rho_s 

# load the era5-land wind data and the bedmodes
chunksize = 64
lonchunks = [45,51]
latchunks = [15,20]

# wind data
dh = xr.open_mfdataset("era5-land_1950_2025_uam10_hist.zarr",engine="zarr",consolidated=False).isel(latitude=slice(latchunks[0]*chunksize,latchunks[1]*chunksize),
                                                                                                    longitude=slice(lonchunks[0]*chunksize,lonchunks[1]*chunksize))

# bed modes (from above)
ds = xr.open_dataset("era5-land_1950_2025_bedmodes_fixed_dm.nc")
ds = ds.reindex(latitude=dh.latitude.values,longitude=dh.longitude.values,fill_value=np.nan,method="nearest",tolerance=0.05) # to align with wind?

# mask of chunks containing dune data (as a function of small_chunksize). Note: Bit redundant for chunksize 1 but versatile if one wants to increase.
small_chunksize=4
llon=len(dh.longitude)//small_chunksize
llat=len(dh.latitude)//small_chunksize
chunkmask=np.zeros((llat,llon))
for j in range(chunkmask.shape[0]):
   for i in range(chunkmask.shape[1]):
       if not np.isnan(ds.da.isel(latitude=slice(j*small_chunksize,(j+1)*small_chunksize),longitude=slice(i*small_chunksize,(i+1)*small_chunksize))).values.all():
           chunkmask[j,i]=1
chunkmask = chunkmask.astype(int)

# test
ds = ds.chunk(dict(longitude=64,latitude=64,ut=64)).persist()

In [ ]:
temp = []
counter = 0
start = time.time()
for j in range(chunkmask.shape[0]):
    for i in range(chunkmask.shape[1]):
        if not chunkmask[j,i]:
            continue

        dhtemp = dh.isel(latitude=slice(j*small_chunksize,(j+1)*small_chunksize),longitude=slice(i*small_chunksize,(i+1)*small_chunksize))
        dstemp = ds.isel(latitude=slice(j*small_chunksize,(j+1)*small_chunksize),longitude=slice(i*small_chunksize,(i+1)*small_chunksize))

        last_start = time.time()
        dhtemp = dhtemp.assign(uf=dhtemp.um_bin*kappa/np.log(z/z0))
        dhtemp = dhtemp.assign(uce=dhtemp.uf*(1+beta*ds.dm*np.abs(np.sin(dhtemp.ua_bin-dstemp.ae))))
        dhtemp = dhtemp.assign(ucb=dhtemp.uf*(1+beta*ds.dm*np.abs(np.sin(dhtemp.ua_bin-dstemp.ab))))
        dhtemp = dhtemp.assign(qf=xr.where(dhtemp.uf>dstemp.ut,(dhtemp.uf**2-dstemp.ut**2)*dstemp.ut*fluxk,0))
        dhtemp = dhtemp.assign(qae=xr.where(dhtemp.uce>dstemp.ut,(dhtemp.uce**2-dstemp.ut**2)*dstemp.ut*fluxk,0))
        dhtemp = dhtemp.assign(qab=xr.where(dhtemp.ucb>dstemp.ut,(dhtemp.ucb**2-dstemp.ut**2)*dstemp.ut*fluxk,0))
        dhtemp = dhtemp.assign(qle=dhtemp.qae*np.cos(dhtemp.ua_bin-dstemp.ae))
        dhtemp = dhtemp.assign(qce=dhtemp.qae*np.sin(dhtemp.ua_bin-dstemp.ae))
        dhtemp = dhtemp.assign(qlb=dhtemp.qab*np.cos(dhtemp.ua_bin-dstemp.ab))
        dhtemp = dhtemp.assign(qcb=dhtemp.qab*np.sin(dhtemp.ua_bin-dstemp.ab))
        dhtemp = dhtemp.assign(dp=dhtemp.qf.weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dhtemp = dhtemp.assign(rdp=(dhtemp.qf*np.exp(1j*dhtemp.ua_bin)).weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        
        dstemp = dstemp.assign(dpm=dhtemp.dp)
        dstemp = dstemp.assign(rdpm=np.abs(dhtemp.rdp))
        dstemp = dstemp.assign(rdpa=np.arctan2(dhtemp.rdp.imag, dhtemp.rdp.real))
        
        dstemp = dstemp.assign(qnle=dhtemp.qle.weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dstemp = dstemp.assign(qale=np.abs(dhtemp.qle).weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dstemp = dstemp.assign(qace=np.abs(dhtemp.qce).weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dstemp = dstemp.assign(qnlb=dhtemp.qlb.weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dstemp = dstemp.assign(qncb=dhtemp.qcb.weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dstemp = dstemp.assign(qalb=np.abs(dhtemp.qlb).weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        dstemp = dstemp.assign(qacb=np.abs(dhtemp.qcb).weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin')))
        
        dstemp = dstemp.assign(at=(xr.where(dstemp.qncb>0,dstemp.ab+np.pi/2,dstemp.ab-np.pi/2)+np.pi)%(2*np.pi)-np.pi) # transverse dune migration direction
        dstemp = dstemp.assign(ao=(xr.where(dstemp.qnlb>0,dstemp.ab,dstemp.ab-np.pi)+np.pi)%(2*np.pi)-np.pi) # oblique dune elongation direction
        
        dstemp = dstemp.assign(sige_sigh=dstemp.qace/dstemp.qacb) # elongating or migrating
        dstemp = dstemp.assign(qha_rdp_dp=dstemp.rdpm/dstemp.dpm) # classic
        dstemp = dstemp.assign(qhab_nl_nc=np.abs(dstemp.qnlb/dstemp.qncb)) # oblique or transverse? sinousity of transverse
        dstemp = dstemp.assign(qhab_ac_nc=np.abs(dstemp.qacb/dstemp.qncb)) # move up or sideways
        dstemp = dstemp.assign(qhne_ac_nl=dstemp.qace/dstemp.qnle) # elongating morphology
        
        dstemp = dstemp.assign(ufm=xr.where(dhtemp.uf>dhtemp.ut,dhtemp.uf/dhtemp.ut,np.nan).weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin'))) # av uflat/uthr where uflat>uthr
        dstemp = dstemp.assign(uft=dhtemp.histogram_um_ua.where(dhtemp.uf>dhtemp.ut).sum(('um_bin','ua_bin'))/dhtemp.histogram_um_ua.sum(('um_bin','ua_bin'))).transpose('ut','latitude','longitude',...) # frac time where uflat>uthr
        dstemp = dstemp.assign(qhbf=dhtemp.qab.weighted(dhtemp.histogram_um_ua.where(dhtemp.uf>dhtemp.ut).fillna(0)).mean(('um_bin','ua_bin'))/dhtemp.qab.weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin'))) # frac crestbed flux where uflat>uthr
        dstemp = dstemp.assign(qhef=dhtemp.qae.weighted(dhtemp.histogram_um_ua.where(dhtemp.uf>dhtemp.ut).fillna(0)).mean(('um_bin','ua_bin'))/dhtemp.qae.weighted(dhtemp.histogram_um_ua).mean(('um_bin','ua_bin'))) # frac crestelong flux where uflat>uthr
        
        dstemp = dstemp[['ab','ae','at','ao','rdpa', # angles
                 'dpm','rdpm', # flat classic
                 'qnlb','qncb','qalb','qacb', # bed fluxes
                 'qnle','qale','qace', # elongate fluxes
                 'qha_rdp_dp','sige_sigh', # 0-1 ratios
                 'qhab_nl_nc','qhab_ac_nc', # bed morphology ratios
                 'qhne_ac_nl', # elongate ratio
                 'ufm','uft', # flat characteristic
                 'qhbf','qhef',  # frac crest flux when flat>thresh
                 'dabdut','daedut', # derivatives bedmodes
                 'da','dm']] # dune orientation and aspect
        counter+=1
        temp.append(dstemp.compute())
        print('\rdune cells: %4d/%4d, time last: %.2f, time total: %6d'%(counter,chunkmask.sum(),time.time()-last_start,time.time()-start),end='')
dsout = xr.combine_by_coords(temp)
dsout = attributes_qsat(dsout) # update attributes
dsout.to_netcdf("era5-land_1950_2025_qsat_fixed_dm.nc")

### Find the best threshold velocity, add dune fields polys and others!

In [ ]:
%%time
ds = xr.open_dataset("../get-ERA5-winds/era5-land_1950_2025_qsat_fixed_dm.nc")
ds = ds.reindex(latitude=dh.latitude.values,longitude=dh.longitude.values,fill_value=np.nan,method="nearest",tolerance=0.05) # to make sure the lat and lon and monotously increasing as previous calculation do not guarantee all non coordinates are included (rerun takes to long)

temp0 = (ds.ae-ds.da+np.pi)%(2*np.pi)-np.pi
temp1 = (np.abs(np.sign(temp0.where(np.abs(temp0)<np.pi/2)).diff('ut'))==2)
temp4 = temp1.cumsum('ut')
temp1root1 = temp1.idxmax('ut') # first root only
# temp1root1 = temp1root1.where(temp1root1>ds.ut.isel(ut=1)) # avoid 0
temp1root2 = temp1.where(temp4 == 2).idxmax('ut') # second root only
temp1 = xr.where((temp1root1<0.2)&(~np.isnan(temp1root2)),temp1root2,temp1root1) # root 1 and 2 if value of first root is below 0.2 and there is a second root use the second else use the first
temp1 = temp1.where(temp1>ds.ut.isel(ut=1)) # avoid effectively 0 m/s ut_opt
temp3 = np.zeros(temp1.shape)*np.nan
for j in range(temp1.shape[0]):
    for i in range(temp1.shape[1]):
        if ~temp1.isel(longitude=i,latitude=j).isnull():
            temp2 = temp0.isel(longitude=i,latitude=j).interp(ut=[temp1.isel(longitude=i,latitude=j)-0.01,temp1.isel(longitude=i,latitude=j)]).to_dataset(name='diffa')
            temp2 = temp2.assign_coords(dict(diffa=temp2.diffa.data))
            temp2['ut'] = (('diffa'),temp2.ut.data)
            temp3[j,i] = temp2.reset_coords().interp(diffa=0).ut.data
ds['ut_opt'] = (("latitude","longitude"),temp3)
ds['ut_opt'].attrs['long_name'] = 'Best-fit threshold friction velocity $u_{\\ast,th_{opt}}$'
ds['ut_opt'].attrs['units'] = 'm/s'
ds['ut_opt'].attrs['description'] = 'Best-fit threshold friction velocity $u_{\\ast,th_{opt}}$ in m/s. For this threshold velocity the obeserved dune orientation $\\alpha_{obs}$ and the elongation mode $\\alpha_{E}$ align. If mutliple solutions exist the first matching threshold velocity was used unless it is below 0.2 m/s and a second match exist in which case the second one was taken.'

In [ ]:
# add dune field mask to the dataset
GVD_shp = gpd.read_file("../Data/Crestlines/GVD.shp")
SIMSTRZ_shp = gpd.read_file("../Data/Crestlines/Simpson_Strz.shp")
GSDGB_shp = gpd.read_file("../Data/Crestlines/GSDGB.shp")

GVD_mask = np.zeros((len(ds.latitude.values),len(ds.longitude.values)))
for i in range(len(ds.longitude.values)):
    for j in range(len(ds.latitude.values)):
        GVD_mask[j,i] = GVD_shp.geometry.iloc[0].contains(Point(ds.longitude.values[i],ds.latitude.values[j]))

SIMSTRZ_mask = np.zeros((len(ds.latitude.values),len(ds.longitude.values)))
for i in range(len(ds.longitude.values)):
    for j in range(len(ds.latitude.values)):
        SIMSTRZ_mask[j,i] = SIMSTRZ_shp.geometry.iloc[0].contains(Point(ds.longitude.values[i],ds.latitude.values[j]))

GSDGB_mask = np.zeros((len(ds.latitude.values),len(ds.longitude.values)))
for i in range(len(ds.longitude.values)):
    for j in range(len(ds.latitude.values)):
        GSDGB_mask[j,i] = GSDGB_shp.geometry.iloc[0].contains(Point(ds.longitude.values[i],ds.latitude.values[j]))

ds["dune_mask"] = xr.where((~np.isnan(ds.da)),1,0)
ds["GSDGB"] = (("latitude","longitude"),GSDGB_mask.astype(int))
ds["SIMSTRZ"] = (("latitude","longitude"),SIMSTRZ_mask.astype(int))
ds["GVD"] = (("latitude","longitude"),GVD_mask.astype(int))

# update the attributes
ds['dune_mask'].attrs['long_name'] = 'Mask of Australian dunes (no coastal dunes).'
ds['dune_mask'].attrs['units'] = 'unitless'
ds['dune_mask'].attrs['description'] = 'Mask of Australian dunes (no coastal dunes).'
ds['GSDGB'].attrs['long_name'] = 'Mask for the Great Sandy and Gibson Desert'
ds['GSDGB'].attrs['units'] = 'unitless'
ds['GSDGB'].attrs['description'] = 'Mask for the Great Sandy and Gibson Desert obtained from hand drawn polygons.'
ds['SIMSTRZ'].attrs['long_name'] = 'Mask for the Simpson and Strzelecki Desert'
ds['SIMSTRZ'].attrs['units'] = 'unitless'
ds['SIMSTRZ'].attrs['description'] = 'Mask for the Simpson and Strzelecki Desert obtained from hand drawn polygons.'
ds['GVD'].attrs['long_name'] = 'Mask for the Great Victoria Desert'
ds['GVD'].attrs['units'] = 'unitless'
ds['GVD'].attrs['description'] = 'Mask for the Great Victoria Desert obtained from hand drawn polygons.'

In [ ]:
# save the whole thing
ds.to_netcdf("era5-land_1950_2025_ut_opt_fixed_dm.nc")